# 🔍 Module 1: Enterprise Advanced RAG & Hybrid Retrieval

Welcome to **Module 1** of the `agentic-ai-playbook`.

✨ **100% Function-Based Architecture**: Zero classes, zero object boilerplate (`self.`), just pure, readable, testable Python functions.

### 🎯 What This Demonstrates:
1. `clean_text()`: Strips noisy decorative ASCII borders (`===`, `---`) before chunking.
2. `chunk_by_sections()` & `chunk_fixed_window()`: Document segmentation.
3. `bm25_search()`: Okapi BM25 sparse keyword ranking.
4. `dense_search()`: Cosine vector semantic similarity.
5. `hybrid_search()`: Reciprocal Rank Fusion (RRF) merging ($RRF(d) = \sum \frac{1}{k + rank_i}$).
6. `rerank_candidates()` & `extract_citations()`: Verifiable citations and grounding.

In [14]:
import sys
import os
import importlib

# Ensure project root is in sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

chunking_mod = importlib.import_module("01_advanced_rag.chunking_strategies")
retriever_mod = importlib.import_module("01_advanced_rag.hybrid_retriever")
reranker_mod = importlib.import_module("01_advanced_rag.reranker")
pipeline_mod = importlib.import_module("01_advanced_rag.rag_pipeline")

print("✅ All Module 1 pure functions imported successfully!")

✅ All Module 1 pure functions imported successfully!


## 1. Clean and Chunk Documents

* **Step 1: Clean Text** - Automatically strips decorative ASCII borders (`===`, `---`) and excessive whitespace so chunks are pure text with zero noise.
* **Step 2: Section Chunking** - Breaks policies into logical sections (`SECTION 1: ...`, `Q01. ...`) preserving complete context.

In [15]:
policy_path = os.path.join("..", "data", "ecommerce", "2026_current_policy.txt")

# load_and_chunk_file automatically runs clean_text() before chunking
fixed_chunks = chunking_mod.load_and_chunk_file(policy_path, strategy="fixed_window", chunk_size=200, overlap=40)
header_chunks = chunking_mod.load_and_chunk_file(policy_path, strategy="header_aware")

print(f"📦 Fixed-Window Chunks : {len(fixed_chunks)} chunks")
print(f"📑 Header-Aware Chunks : {len(header_chunks)} clean semantic sections")
print(f"\nSample Cleaned Section (Notice: Zero '===' borders):\n{header_chunks[0]['text'][:300]}...")

📦 Fixed-Window Chunks : 7 chunks
📑 Header-Aware Chunks : 6 clean semantic sections

Sample Cleaned Section (Notice: Zero '===' borders):
NOVATECH ELECTRONICS: GLOBAL STORE POLICY & CUSTOMER CARE HANDBOOK

Document ID      : DOC-POL-2026-V2
Document Status  : CURRENT // EFFECTIVE MARCH 1, 2026
Document Version : 2026.2 (Replaces all previous 2024 and 2025 editions)
Support Contact  : support@novatech-electronics.com | Toll-Free: +1-80...


## 2. Hybrid Search (Keywords + Meaning Combined)

**Why use both?**
* **Keyword Search (BM25)** catches exact words, product names, SKU codes, and specific numbers.
* **Meaning Search (Dense)** understands similar ideas even if phrased differently.
* We combine the top results from both so we get the best of both worlds!

In [16]:
query = "What is the return window for laptops and drones?"

# Call hybrid_search function directly
top_results = retriever_mod.hybrid_search(query, header_chunks, top_k=3)

print(f"🔎 Query: '{query}'\n")
for r in top_results:
    print(f"🏆 [Chunk ID: {r['chunk_id']}] | Combined Score: {r['rrf_score']} | Keyword Rank: {r['bm25_rank']} | Meaning Rank: {r['dense_rank']}")
    print(f"   Text snippet: {r['text'][:150]}...\n")

🔎 Query: 'What is the return window for laptops and drones?'

🏆 [Chunk ID: 2026_current_policy.txt_sec_001] | Combined Score: 0.03252 | Keyword Rank: 1 | Meaning Rank: 2
   Text snippet: SECTION 1: RETURNS, REFUNDS & REPLACEMENTS POLICY

1.1 Standard 30-Day Return Window:
    Customers may return eligible items within 30 calendar days ...

🏆 [Chunk ID: 2026_current_policy.txt_sec_003] | Combined Score: 0.03227 | Keyword Rank: 3 | Meaning Rank: 1
   Text snippet: SECTION 3: WARRANTY COVERAGE & REPAIR SERVICES

3.1 Standard 1-Year Limited Hardware Warranty:
    Every new device purchased directly from NovaTech i...

🏆 [Chunk ID: 2026_current_policy.txt_sec_004] | Combined Score: 0.03151 | Keyword Rank: 2 | Meaning Rank: 5
   Text snippet: SECTION 4: NOVACLUB VIP LOYALTY & REWARDS PROGRAM

NovaClub is our customer rewards tier system designed to give frequent shoppers discounts, extended...



In [27]:
print(citations[0])

{'chunk_id': '2026_current_policy.txt_sec_001', 'source_file': '2026_current_policy.txt', 'section': 'SECTION 1: RETURNS, REFUNDS & REPLACEMENTS POLICY', 'quote': '1.2 Electronics & High-Value Exceptions (14-Day Limit):\n    Laptops, desktop computers, digital cameras, drones, and smartphones have a restricted 14-calendar-day return\n    window...'}


## 3. Re-Rank Results & Pick Exact Quote Citations

Filters the results to the top 2 best matches and grabs the exact sentence to prove where the answer came from.

In [ ]:
top_reranked = reranker_mod.rerank_candidates(query, top_results, top_n=2)

citations = reranker_mod.extract_citations(query, top_reranked)

print("📌 Grounded Citations:")
for cit in citations:
    print(f"* File: {cit['source_file']} | Section: {cit['section']}")
    print(f"  Quote: \"{cit['quote']}\"\n")

📌 Grounded Citations:
* File: 2026_current_policy.txt | Section: SECTION 1: RETURNS, REFUNDS & REPLACEMENTS POLICY
  Quote: "1.2 Electronics & High-Value Exceptions (14-Day Limit):
    Laptops, desktop computers, digital cameras, drones, and smartphones have a restricted 14-calendar-day return
    window..."

* File: 2026_current_policy.txt | Section: SECTION 3: WARRANTY COVERAGE & REPAIR SERVICES
  Quote: "3.2 What the Standard Warranty Covers:
    * Internal motherboard, CPU, and GPU hardware component failures."



## 4. Ask a Question (Complete RAG in 1 Call)

Runs the entire pipeline in one simple function: Search -> Re-Rank -> Grounded Answer with Citations.

In [28]:
data_directory = os.path.join("..", "data", "ecommerce")
all_chunks = pipeline_mod.load_knowledge_base(data_dir=data_directory)

response = pipeline_mod.ask_rag("Can I price match a competitor item?", chunks=all_chunks)

print(f"❓ Question: {response['query']}\n")
print(f"💡 Answer:\n{response['answer']}\n")
print(f"⚡ Latency: {response['latency_ms']} ms")

❓ Question: Can I price match a competitor item?

💡 Answer:
Based on verified documentation [store_faq_kb.txt | Q03. What is the price match guarantee policy?]:
"What is the price match guarantee policy?"

Verified against 2 source citations.

⚡ Latency: 4589.3 ms


In [33]:
import json
print(json.dumps(response, indent=4))

{
    "query": "Can I price match a competitor item?",
    "answer": "Based on verified documentation [store_faq_kb.txt | Q03. What is the price match guarantee policy?]:\n\"What is the price match guarantee policy?\"\n\nVerified against 2 source citations.",
    "citations": [
        {
            "chunk_id": "store_faq_kb.txt_sec_003",
            "source_file": "store_faq_kb.txt",
            "section": "Q03. What is the price match guarantee policy?",
            "quote": "What is the price match guarantee policy?"
        },
        {
            "chunk_id": "store_faq_kb.txt_sec_005",
            "source_file": "store_faq_kb.txt",
            "section": "Q05. How do I return a bundle deal if I only want to keep one item?",
            "quote": "If a customer retains individual components from a bundle, the retained item is billed at its full standalone\n     MSRP price, and only the remaining net difference is refunded."
        }
    ],
    "retrieved_chunks": [
        {
     